In [ ]:
# 필요한 패키지 설치
!pip install highway-env stable-baselines3[extra]

In [ ]:
# 고속도로 환경 가져오기
import gymnasium as gym
import highway_env

# Stable Baseline3 가져오기
from stable_baselines3 import DQN
import numpy as np

# 결과값 시각화를 위한 라이브러리 가져오기
from IPython import display
import matplotlib.pyplot as plt
from gymnasium.wrappers import RecordVideo
import os
from IPython.display import HTML
from base64 import b64encode
import glob

## 상태 (Observation) 공간
이 환경의 상태는 연속적인 타입으로 구성되어 있으며 다음과 같습니다.

    관찰 차량 수: `vehicles_count`
    각 차량마다 다음 특성들이 관찰됨:
      - presence: 차량 존재 여부 (1: 존재, 0: 없음)
      - x, y: 위치 좌표
      - vx, vy: 속도 벡터
      - cos_h, sin_h: 방향 벡터

## 액션 (Action) 공간
5개의 이산적인 행동이 가능합니다:

    0: LANE_LEFT (왼쪽 차선 변경)
    1: IDLE (현재 상태 유지)
    2: LANE_RIGHT (오른쪽 차선 변경)
    3: FASTER (가속)
    4: SLOWER (감속)

## 종료 조건
에피소드는 다음 조건에서 종료됩니다:

**일반 종료 (terminated)**

    설정된 duration에 도달했을 때
    차량 충돌이 발생했을 때
    도로를 이탈했을 때

**즉시 종료 (truncated)**

    최대 허용 속도를 초과했을 때
    최소 속도 미만으로 주행할 때
    차량의 물리적 가속/감속 한계를 넘어섰을 때
    물리적으로 불가능한 급격한 차선 변경을 시도할 때

In [ ]:
# 환경 생성 및 설정
env = gym.make("highway-v0", render_mode='rgb_array')
env.unwrapped.configure({
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,
    },
    "action": {
        "type": "DiscreteMetaAction",
    },
    "duration": 40,  # 에피소드 길이
    "lanes_count": 5,
    "vehicles_count": 20,  # 다른 차량 수
    "screen_width": 600,  # 렌더링 화면 크기
    "screen_height": 150,
})

In [ ]:
# 모델 생성
model = DQN(
    "MlpPolicy",
    env,
    learning_rate=5e-4,
    buffer_size=15000,
    learning_starts=200,
    batch_size=32,
    gamma=0.8,
    train_freq=1,
    gradient_steps=1,
    target_update_interval=50,
    verbose=0
)

# 모델 학습
model.learn(total_timesteps=1000)

In [ ]:
def show_video():
    """
    비디오 파일을 찾아서 표시하는 함수
    """
    # 생성된 mp4 파일 찾기
    video_files = glob.glob("./highway-agent-episode-*.mp4")
    if not video_files:
        print("No video files found")
        return

    # 가장 최근 비디오 파일 사용
    latest_video = max(video_files, key=os.path.getctime)

    # 비디오 표시
    video = open(latest_video, 'rb').read()
    html = '''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(b64encode(video).decode())
    return HTML(html)

In [ ]:
# 비디오 저장을 위한 평가 환경 생성
eval_env = gym.make("highway-v0", render_mode='rgb_array')
eval_env = RecordVideo(eval_env, video_folder="./", name_prefix="highway-agent")

# 평가를 위해 에피소드 1개 실행하고 그 비디오 저장
obs, _ = eval_env.reset()
for _ in range(100):
    action, _ = model.predict(obs, deterministic=True)
    obs, _, done, truncated, _ = eval_env.step(action)
    if done or truncated:
        break

eval_env.close()

In [ ]:
# 저장된 비디오 표시
show_video()